# Heart Failure Clinical Records - Classification ML Assignment

## Complete Preprocessing, Feature Scaling, Imbalance Handling, and Model Comparison

### Dataset Description
- **299 records** of heart failure clinical patients
- **13 features**: age, anaemia, creatinine_phosphokinase, diabetes, ejection_fraction, high_blood_pressure, platelets, serum_creatinine, serum_sodium, sex, smoking, time
- **Target**: DEATH_EVENT (1 = death, 0 = survival)
- **Class imbalance**: 67.9% survival vs 32.1% death - dataset is imbalanced
- **No missing values**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.metrics import classification_report_imbalanced

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                           classification_report, confusion_matrix, ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### 1. Load and Explore the Dataset

In [4]:
df = pd.read_csv('heart_failure_clinical_records_dataset.csv')
display(df.head())

print('Shape:', df.shape)

print('\nClass distribution:')
print(df['DEATH_EVENT'].value_counts())

print('\nMissing values:')
print(df.isnull().sum())

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


Shape: (299, 13)

Class distribution:
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

Missing values:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64


### 2. Basic Visualization

In [1]:
sns.countplot(x='DEATH_EVENT', data=df)
plt.title('Class Distribution: DEATH_EVENT')
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

NameError: name 'sns' is not defined

### 3. Preprocessing Pipeline
- **Feature Scaling**: StandardScaler (z-score normalization)
- **Imbalance Handling**: SMOTE (Synthetic Minority Over-sampling Technique)
- Train/Test split: 80/20 with stratification

In [4]:
X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']

# Identify numeric and categorical columns
numeric_features = X.columns.tolist()
categorical_features = []

# All features are numeric in this dataset
print('Numeric features:', numeric_features)

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Train class distribution: {np.bincount(y_train)}')
print(f'Test class distribution: {np.bincount(y_test)}')

### 4. Model Comparison with Pipeline (Scaling + SMOTE)

We compare multiple classification models using a pipeline that includes:
1. Feature Standardization
2. SMOTE for class balancing
3. Classification model

In [5]:
def evaluate_model(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    }

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', probability=True, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

# Evaluate each model
results = []
for name, model in models.items():
    # Create pipeline with scaler and SMOTE
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42, k_neighbors=2)),
        ('classifier', model)
    ])
    
# Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    
# Fit on training data
    pipeline.fit(X_train, y_train)
    
# Predict
    y_pred = pipeline.predict(X_test)
    metrics = evaluate_model(y_test, y_pred)
    
    result = {
        'model': name,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        **metrics
    }
    results.append(result)
    print(f'{name}:')
    print(f'  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
    print(f'  Test Accuracy: {metrics["accuracy"]:.4f}')
    print(f'  Precision: {metrics["precision"]:.4f}')
    print(f'  Recall: {metrics["recall"]:.4f}')
    print(f'  F1-Score: {metrics["f1"]:.4f}')
    print()

### 5. Detailed Classification Reports and Confusion Matrices

In [6]:
best_model_name = results[0]['model']
best_score = results[0]['accuracy']

for result in results:
    print(f'\n=== {result["model"]} ===')
    print(f'Classification Report:')
    print(classification_report(y_test, y_pred))
    
# Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Survived (0)', 'Death (1)']
    )
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix: {result["model"]}')
    plt.show()

    print(f'Accuracy: {result["accuracy"]:.4f}')
    print(f'Precision: {result["precision"]:.4f}')
    print(f'Recall: {result["recall"]:.4f}')
    print(f'F1-Score: {result["f1"]:.4f}')

    if result['accuracy'] > best_score:
        best_score = result['accuracy']
        best_model_name = result['model']